### 1. The Ridge Loss Function

We start with the standard Mean Squared Error (MSE) loss function, and add our L2 penalty (the sum of the squared weights, multiplied by our hyperparameter $\alpha$).

$$L(W, b) = \frac{1}{N} \sum_{i=1}^N (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^m w_j^2$$

### 2. The Gradients (Partial Derivatives)

To navigate down this new, penalized loss bowl, we take the derivative.

#### A. Derivative with respect to the Intercept ($b$):

Remember the golden rule: we never penalize the intercept. Therefore, the derivative for $b$ is exactly the same as standard Gradient Descent. The penalty term completely vanishes.

$$\frac{\partial L}{\partial b} = -\frac{2}{N} \sum_{i=1}^N (y_i - \hat{y}_i)$$

#### B. Derivative with respect to the Weights ($W$):

This is where the magic happens. We take the derivative of the MSE, and add the derivative of the penalty term ($\alpha w^2$), which is simply $2\alpha w$.

In vectorized matrix form for all weights:

$$\nabla_W L = \left[ -\frac{2}{N} X^T (Y - \hat{Y}) \right] + 2\alpha W$$

### 3. The Update Rule (Weight Decay)

Let's plug our new penalized gradient into the standard Gradient Descent update rule:

$$W_{new} = W_{old} - \eta \left( -\frac{2}{N} X^T(Y - \hat{Y}) + 2\alpha W_{old} \right)$$

If we distribute the learning rate ($\eta$) and rearrange the terms, we expose one of the most famous concepts in Deep Learning:

$$W_{new} = W_{old} - 2\eta\alpha W_{old} + \eta \frac{2}{N} X^T(Y - \hat{Y})$$

$$W_{new} = \mathbf{W_{old}(1 - 2\eta\alpha)} + \eta \frac{2}{N} X^T(Y - \hat{Y})$$

### The Engineering Takeaway

Look at the bolded section: $\mathbf{W_{old}(1 - 2\eta\alpha)}$.

Because $\eta$ and $\alpha$ are positive numbers, $(1 - 2\eta\alpha)$ acts as a fractional multiplier (like $0.99$). Before the algorithm even looks at the data to take a step, it physically shrinks the existing weight by 1%. In deep learning, L2 Regularization is officially called Weight Decay because of this exact equation. The weights naturally decay toward zero on every single epoch unless the data strongly pulls them back up.

# Ridge Regression implementation with the gradient descent technique:

In [5]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge

In [3]:
# generating dataset
# 500 samples, 5 features, with heavy noise
X, y = make_regression(n_samples=500, n_features=5, noise=15.0, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling is MANDATORY for Gradient Descent and Ridge Regularization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
# custom ridge with gd
class RidgeGD:
  def __init__(self, alpha=1.0, learning_rate=0.1, epochs=1000):
    self.alpha = alpha
    self.lr = learning_rate
    self.epochs = epochs
    self.coef_ = None
    self.intercept_ = None

  def fit(self, X, y):
    N, m = X.shape
    self.coef_ = np.zeros(m)
    self.intercept_ = 0.0

    for i in range(self.epochs):
      y_pred = np.dot(X, self.coef_) + self.intercept_

      # error
      error = y_pred - y

      # calculate gradients: intercept graadient (no penalty)
      dl_db = (2/N) * np.sum(error)

      # weight gradient (adding the l2 penalty derivative )
      # Note: We divide 2*alpha by N to mathematically align Mean Squared Error with Scikit-Learn's SSE
      dl_dw = (2/N) * np.dot(X.T, error) + (2 * self.alpha / N) * self.coef_

      # update weights
      self.intercept_ -= self.lr * dl_db
      self.coef_ -= self.lr * dl_dw

  def predict(self, X):
    return np.dot(X, self.coef_) + self.intercept_



In [10]:
# benchmark custom gd vs scikit learn
alpha_val = 50.0  # high penalty to force shrinkage

# Train Custom Ridge GD
custom_ridge_gd = RidgeGD(alpha=alpha_val, learning_rate=0.1, epochs=500)
custom_ridge_gd.fit(X_train_scaled, y_train)

# Train Scikit-Learn Ridge (Closed-form solver)
sk_ridge = Ridge(alpha=alpha_val)
sk_ridge.fit(X_train_scaled, y_train)

print(f"RIDGE BENCHMARK (Alpha = {alpha_val})")

print("\n1. R2 SCORES (TEST DATA)")
print(f"Custom Ridge GD: {r2_score(y_test, custom_ridge_gd.predict(X_test_scaled)):.6f}")
print(f"Sklearn Ridge:   {r2_score(y_test, sk_ridge.predict(X_test_scaled)):.6f}")

print("\n2. COEFFICIENTS COMPARISON (First 3)")
print(f"Custom GD:     {custom_ridge_gd.coef_[:3]}")
print(f"Sklearn:       {sk_ridge.coef_[:3]}")

print("\n3. INTERCEPT COMPARISON")
print(f"Custom GD:     {custom_ridge_gd.intercept_:.6f}")
print(f"Sklearn:       {sk_ridge.intercept_:.6f}")

RIDGE BENCHMARK (Alpha = 50.0)

1. R2 SCORES (TEST DATA)
Custom Ridge GD: 0.970381
Sklearn Ridge:   0.970381

2. COEFFICIENTS COMPARISON (First 3)
Custom GD:     [24.44903154 68.46047103 27.36009239]
Sklearn:       [24.44903154 68.46047103 27.36009239]

3. INTERCEPT COMPARISON
Custom GD:     3.281604
Sklearn:       3.281604


If you look at the output when you run this, our iterative for loop (Gradient Descent) perfectly hits the exact same target that Scikit-Learn hits using advanced matrix algebra.

We didn't invert a single matrix. We just applied Weight Decay, letting the algorithm shave a tiny fraction off the weights on every step, gracefully pulling them down to a stabilized minimum. If this data had 10 million rows, Scikit-Learn's default Matrix method would crash your computer, but your Custom GD method would solve it effortlessly.